# ASG Airlines - pipeline walkthrough

Every transformation shown here is imported from `src/`; nothing is reimplemented in this
notebook. Run `python -m src.pipeline` first so the gold layer and warehouse exist.

In [1]:
import sys

import duckdb
import pandas as pd

sys.path.insert(0, "..")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

from src.clean import clean, parse_source_duration
from src.config import load_config
from src.ingest import ingest
from src.model import build_model
from src.validate import validate

config = load_config()
bronze = ingest(config, run_id="notebook")
{name: df.shape for name, df in bronze.items()}

{'flights': (1020, 11),
 'bookings': (1000, 13),
 'payments': (1000, 8),
 'passengers': (1039, 13)}

## The duration column holds two Python types

1,019 values are `datetime.time`. One is `datetime.datetime(1899, 12, 29, 5, 0)` - Excel's
representation of a negative duration. `pd.to_timedelta` raises on it.

In [2]:
raw_flights = pd.read_excel(config.paths.workbook, "flights")
raw_flights["duration"].map(lambda v: type(v).__name__).value_counts()

duration
time        1019
datetime       1
Name: count, dtype: int64

In [3]:
odd = raw_flights[raw_flights["duration"].map(lambda v: type(v).__name__) != "time"]
odd[["flight_id", "departure_time", "arrival_time", "duration"]]

,flight_id,departure_time,arrival_time,duration
355,SJ192,2026-04-19 18:45:42,2026-04-18 23:45:42,1899-12-29 05:00:00


## SJ192 reconciles rather than guesses

Its arrival lands a day before its departure: -1,140 minutes. Rolling the arrival forward one
day gives 300 minutes, which is exactly what the raw `duration` column already says.

In [4]:
sj192_raw = raw_flights[raw_flights["flight_id"] == "SJ192"].iloc[0]
naive_minutes = (sj192_raw["arrival_time"] - sj192_raw["departure_time"]).total_seconds() / 60
pd.Series(
    {
        "departure_time": sj192_raw["departure_time"],
        "arrival_time": sj192_raw["arrival_time"],
        "naive_duration_minutes": naive_minutes,
        "source_duration_column": sj192_raw["duration"],
        "source_duration_minutes": parse_source_duration(sj192_raw["duration"]),
        "delta_minutes": parse_source_duration(sj192_raw["duration"]) - naive_minutes,
    }
)

departure_time             2026-04-19 18:45:42
arrival_time               2026-04-18 23:45:42
naive_duration_minutes                 -1140.0
source_duration_column     1899-12-29 05:00:00
source_duration_minutes                  300.0
delta_minutes                           1440.0
dtype: object

In [5]:
validated, rule_results = validate(bronze)
cleaned = clean(validated, config)

flights = cleaned.silver["flights"]
flights[flights["flight_id"] == "SJ192"][
    ["departure_ts", "arrival_ts", "duration_minutes", "source_duration_minutes",
     "was_corrected", "correction_reason", "is_overnight", "is_red_eye"]
]

,departure_ts,arrival_ts,duration_minutes,source_duration_minutes,was_corrected,correction_reason,is_overnight,is_red_eye
355,2026-04-19 18:45:42,2026-04-19 23:45:42,300.0,300.0,True,arrival_rolled_forward_one_day,False,False


The reconciliation is reported for every row, not just this one: exactly one mismatch before
the correction, off by exactly one day, and none after.

In [6]:
pd.Series(cleaned.reconciliation)

source_parseable              1020.000
mismatch_before_correction       1.000
mismatch_after_correction        0.000
max_abs_delta_before          1440.000
max_abs_delta_after              0.005
dtype: float64

## Duplicate keys, two different correct treatments

15 of the 16 duplicated `flight_id`s are byte-identical rows and collapse to one. `6F250`
disagrees on source city and timestamps, so no survivor can be justified and the group is
quarantined. `drop_duplicates()` would have silently kept whichever row came first.

In [7]:
dupes = raw_flights[raw_flights["flight_id"].duplicated(keep=False)]
pd.DataFrame(
    {
        "duplicated_flight_ids": [dupes["flight_id"].nunique()],
        "exact_duplicate_rows_removed": [cleaned.anomalies["exact_duplicate"]],
        "quarantined_rows": [cleaned.anomalies["conflicting_duplicate_key"]],
    }
)

,duplicated_flight_ids,exact_duplicate_rows_removed,quarantined_rows
0,16,15,2


In [8]:
raw_flights[raw_flights["flight_id"] == "6F250"][
    ["flight_id", "airline", "source", "destination", "departure_time", "arrival_time"]
]

,flight_id,airline,source,destination,departure_time,arrival_time
253,6F250,UNKNOWN,DEL,BLR,2026-04-20 03:26:41.701,2026-04-20 07:30:41.701
270,6F250,UNKNOWN,CCU,BLR,2026-04-20 02:23:41.702,2026-04-20 02:56:41.702


## Passenger duplicates need a survivorship rule, not a drop

All 36 duplicated passenger ids conflict on email, phone, Aadhaar and date of birth, and none
is a byte-identical row. The rule is: fewest nulls, then a full 12-digit Aadhaar, then the
lowest email.

In [9]:
log = cleaned.survivorship
log[log["passenger_id"] == log["passenger_id"].iloc[0]]

,passenger_id,candidate_row,null_count,aadhaar_digits,email_masked,survived,reason
0,P1034,34,0,12,i***r@outlook.com,False,
1,P1034,35,0,12,i***r@gmail.com,True,lowest_email


In [10]:
log[log["survived"]]["reason"].value_counts()

reason
lowest_email           21
fewest_nulls           10
aadhaar_full_length     5
Name: count, dtype: int64

## Aadhaar lost its leading zeros to int64 storage

114 of 1,039 values are shorter than 12 digits. Hashing before zero-padding would give the same
person two different tokens, so `zfill(12)` runs first and the HMAC is keyed with a pepper.

In [11]:
raw_passengers = pd.read_excel(config.paths.workbook, "passengers")
raw_passengers["aadhaar_id"].astype(str).str.len().value_counts().sort_index()

aadhaar_id
10      5
11    109
12    925
Name: count, dtype: int64

In [12]:
from src.pii import hmac_token, passenger_surrogate_key

pd.DataFrame(
    [
        {"input": "7345678901", "digits": 10,
         "token_with_zfill": passenger_surrogate_key("7345678901", "demo-pepper"),
         "token_without_zfill": hmac_token("7345678901", "demo-pepper")},
        {"input": "007345678901", "digits": 12,
         "token_with_zfill": passenger_surrogate_key("007345678901", "demo-pepper"),
         "token_without_zfill": hmac_token("007345678901", "demo-pepper")},
    ]
)

,input,digits,token_with_zfill,token_without_zfill
0,7345678901,10,PSG_0303d78218e580df,63aeab742a38e797
1,007345678901,12,PSG_0303d78218e580df,0303d78218e580df


## payments.amount carries three types and two distinct failures

30 rows hold the literal string `"INVALID"` and 48 are null. `to_numeric(errors="coerce")`
turns both into NaN, which hides the difference between a value that was never captured and one
that was captured wrongly.

In [13]:
raw_payments = pd.read_excel(config.paths.workbook, "payments")
pd.DataFrame(
    {
        "python_types": [raw_payments["amount"].map(lambda v: type(v).__name__).value_counts().to_dict()],
        "literal_INVALID": [(raw_payments["amount"] == "INVALID").sum()],
        "nulls": [raw_payments["amount"].isna().sum()],
    }
).T

,0
python_types,"{'float': 961, 'str': 30, 'int': 9}"
literal_INVALID,30
nulls,48


In [14]:
cleaned.silver["payments"]["amount_quality"].value_counts()

amount_quality
valid          922
missing         48
non_numeric     30
Name: count, dtype: int64

## The fan-out that inflates revenue by 208,615.94

16 duplicate flight ids and 1,000 payments across 637 bookings turn a 1,000-row booking table
into 1,404 rows. Summing `amount` on that join double-counts.

In [15]:
naive = raw_flights.pipe(
    lambda f: pd.read_excel(config.paths.workbook, "bookings").merge(f, on="flight_id", how="left")
).merge(raw_payments, on="booking_id", how="left")

true_sum = pd.to_numeric(raw_payments["amount"], errors="coerce").sum()
naive_sum = pd.to_numeric(naive["amount"], errors="coerce").sum()
pd.Series(
    {
        "booking_rows": 1000,
        "rows_after_triple_join": len(naive),
        "naive_sum": round(naive_sum, 2),
        "true_sum": round(true_sum, 2),
        "inflation": round(naive_sum - true_sum, 2),
        "inflation_pct": round(100 * (naive_sum - true_sum) / true_sum, 2),
    }
)

booking_rows                 1000.00
rows_after_triple_join       1404.00
naive_sum                 7593758.92
true_sum                  7385142.98
inflation                  208615.94
inflation_pct                   2.82
dtype: float64

Keeping payments at payment grain in `fact_payment`, joined no further than `fact_booking`,
reproduces the true figure.

In [16]:
model = build_model(cleaned.silver, config)
pd.Series(
    {
        "fact_booking_rows": len(model["fact_booking"]),
        "fact_payment_rows": len(model["fact_payment"]),
        "fact_payment_sum": round(model["fact_payment"]["amount"].sum(), 2),
    }
)

fact_booking_rows       1000.00
fact_payment_rows       1000.00
fact_payment_sum     7385142.98
dtype: float64

## KPIs from the warehouse

Delay is absent by design: the workbook has no scheduled-versus-actual pair, so an anomaly rate
is reported instead of an invented on-time performance figure.

In [17]:
con = duckdb.connect(str(config.paths.warehouse), read_only=True)
con.execute("SELECT * FROM v_kpi_headline").df().T.rename(columns={0: "value"})

,value
flights,1003.000
bookings,1000.000
passengers,1000.000
avg_duration_minutes,164.670
stddev_duration_minutes,77.390
gross_revenue,7385142.980
confirmed_revenue,2471402.040
cancellation_rate_pct,31.400
confirmation_rate_pct,32.000
payment_coverage_pct,63.700


In [18]:
con.execute("SELECT * FROM v_flights_by_airline").df()

,airline_name,is_unknown,flights,share_pct,avg_duration_minutes,stddev_duration_minutes
0,IndiGo,False,249,24.83,167.70,80.72
1,SpiceJet,False,236,23.53,164.19,77.43
2,Air India,False,233,23.23,163.15,74.21
3,Vistara,False,218,21.73,162.68,77.99
4,UNKNOWN,True,67,6.68,166.90,75.36


In [19]:
con.execute(
    """
    SELECT route_label, flights, bookings, avg_duration_minutes, stddev_duration_minutes,
           min_duration_minutes, max_duration_minutes, revenue
    FROM v_route_performance
    ORDER BY bookings DESC
    LIMIT 10
    """
).df()

,route_label,flights,bookings,avg_duration_minutes,stddev_duration_minutes,min_duration_minutes,max_duration_minutes,revenue
0,BOM-CCU,90,87,169.51,72.02,33.0,293.0,700135.30
1,CCU-DEL,72,73,153.61,76.96,32.0,298.0,582851.37
2,MAA-BLR,65,64,172.82,79.97,34.0,300.0,492102.05
3,BLR-BOM,60,62,147.73,73.89,35.0,293.0,506069.04
4,HYD-MAA,57,57,152.81,78.42,35.0,298.0,438428.50
5,DEL-HYD,54,55,174.76,75.97,35.0,290.0,521798.78
6,HYD-DEL,42,41,185.36,78.64,35.0,291.0,305771.13
7,BOM-DEL,39,36,153.82,75.28,35.0,290.0,299355.97
8,CCU-BOM,33,34,163.97,81.22,35.0,287.0,150839.53
9,DEL-BOM,28,29,178.11,82.99,43.0,286.0,191097.12


Route duration is reported with its spread. Within BOM-CCU alone durations run 33 to 293
minutes, so a route average describes this data rather than a scheduled block time.

In [20]:
con.execute("SELECT * FROM anomaly_summary ORDER BY count DESC").df()

,anomaly,scope_table,count,scope_rows,rate_pct
0,booking_without_payment,bookings,363,1000,36.300
1,aadhaar_length_anomaly,passengers,114,1039,10.972
2,duplicate_passenger_id,passengers,75,1039,7.218
3,missing_amount,payments,48,1000,4.800
4,missing_status,bookings,45,1000,4.500
5,missing_airline,flights,41,1020,4.020
6,sentinel_airline,flights,31,1020,3.039
7,non_numeric_amount,payments,30,1000,3.000
8,sentinel_status,bookings,30,1000,3.000
9,flight_without_booking,flights,20,1020,1.961


In [21]:
con.execute("SELECT * FROM dq_score").df()

,table,ingested,flagged,quarantined,collapsed_duplicates,retained,dq_score
0,flights,1020,68,2,15,1003,0.9314
1,bookings,1000,75,0,0,1000,0.9250
2,payments,1000,78,0,0,1000,0.9220
3,passengers,1039,107,0,39,1000,0.8970


In [22]:
result = con.execute("SELECT * FROM v_bookings_monthly").df()
con.close()
result

,year,month,month_name,bookings,confirmed,cancelled
0,2025,4,April,48,14.0,16.0
1,2025,5,May,80,27.0,26.0
2,2025,6,June,89,25.0,30.0
3,2025,7,July,66,24.0,21.0
4,2025,8,August,101,29.0,32.0
5,2025,9,September,75,22.0,26.0
6,2025,10,October,76,34.0,18.0
7,2025,11,November,90,34.0,27.0
8,2025,12,December,89,29.0,32.0
9,2026,1,January,89,26.0,30.0
